In [11]:
from tokenizers import Tokenizer
from tokenizers.models import BPE, WordPiece, Unigram
from tokenizers.pre_tokenizers import BertPreTokenizer, ByteLevel
from tokenizers.trainers import BpeTrainer, WordPieceTrainer, UnigramTrainer
from pathlib import Path
from typing import List, Literal, Optional
from tokenizers.processors import ByteLevel as ByteLevelProcessor
from tokenizers.normalizers import Sequence, NFKC
from pathlib import Path
from typing import List, Literal, Optional
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from tokenizers.models import Unigram
from tokenizers.trainers import UnigramTrainer
from helper_functions import load_wikipedia_text
import numpy as np

In [12]:
# https://arxiv.org/pdf/2203.15556#page=24.27  in this paper they set the ratio for optimal language model training to 20 tokens per trainable parameter
trainable_params = 2_221_696
TARGET = trainable_params * 20  # target number of characters to load; we fix this for all other experiments

# FIXME: the target does not takeinot account the 70/10/20 split between train/val/test

text_en = load_wikipedia_text("en", TARGET)
text_ru = load_wikipedia_text("ru", TARGET)
text =  text_ru + text_en


Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/41 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/21 [00:00<?, ?it/s]

In [13]:
# --------------------------------------------------
#             Custom tokenizer training
# --------------------------------------------------


TokenizerModel = Literal["bpe", "wordpiece", "unigram", "bytelevel"]

def train_custom_tokenizer(
    texts: List[str], 
    model_type: TokenizerModel, 
    vocab_size: int, 
    min_frequency: int = 2,
    special_tokens: Optional[List[str]] = None,
    save_path: str = f"tokenizers/custom_tokenizer.json",
):

    if special_tokens is None:
        special_tokens = [ "[UNK]", "[EOS]"]

    print(f"Training {model_type.upper()} tokenizer...")
    
    """
    args:
        texts: List of strings to train the tokenizer on
        model_type: Type of tokenizer to train ("bpe", "wordpiece", "unigram", "bytelevel")
        vocab_size: Size of the vocabulary
        min_frequency: Minimum frequency for a token to be included in the vocabulary (only for BPE and WordPiece)
        special_tokens: List of special tokens to include in the vocabulary
        save_path: Path to save the trained tokenizer
    returns:
        Trained tokenizer object (ids and tokens)
    """

    # --------------------------
    # Normal BPE (Unicode tokens)
    # --------------------------
    if model_type == "bpe":
        tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
        tokenizer.normalizer = Sequence([NFKC()])
        tokenizer.pre_tokenizer = BertPreTokenizer()

        trainer = BpeTrainer(
            vocab_size=vocab_size,
            min_frequency=min_frequency,
            special_tokens=special_tokens,
        )

    # --------------------------
    # BYTE-LEVEL BPE  Tokenizer
    # --------------------------
    # here we observe issues with encoding decoding the russina text
    # uses raw utf-8 bytes and merges are applied to byte sequences, not charcaters like before. Generally, this is used to avoud UNK characters
    # the issue with russian tokens is the pre-processing and post-processing, and the fact that on reconstruction, so garbage characters are formmed. 
    # Пр → corrupted → ÐŁÑĢ
    
    elif model_type == "bytelevel":
        tokenizer = Tokenizer(BPE(unk_token="[UNK]"))

        # must operate on bytes BEFORE unicode interpretation
        tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=True)

        tokenizer.normalizer = None

        ## must be used for proper decoding back into russian characters
        tokenizer.decoder = ByteLevelDecoder()

        # must reconstruct bytes AFTER merging
        tokenizer.post_processor = ByteLevelProcessor()

        trainer = BpeTrainer(
            vocab_size=vocab_size,
            special_tokens=special_tokens
        )
    # --------------------------
    #     Unigram Tokenizer
    # --------------------------
    elif model_type == "unigram":
        tokenizer = Tokenizer(Unigram())
        
        tokenizer.normalizer = Sequence([NFKC()]) 
        tokenizer.pre_tokenizer = BertPreTokenizer()
        
        trainer = UnigramTrainer(
            vocab_size=vocab_size,
            special_tokens=special_tokens,
            unk_token="[UNK]"
        )

    # --------------------------
    #    Wordpiece Tokenizer
    # --------------------------
    elif model_type == "wordpiece":
        tokenizer = Tokenizer(WordPiece(unk_token="[UNK]"))

        tokenizer.normalizer = Sequence([NFKC()])
        tokenizer.pre_tokenizer = BertPreTokenizer()

        trainer = WordPieceTrainer(
            vocab_size=vocab_size,
            min_frequency=min_frequency,
            special_tokens=special_tokens,
        )

    # --------------------------
    #       Else error ...
    # --------------------------
    else:
        raise ValueError(f"Unknown model type: {model_type}")



    # Train tokenizer
    tokenizer.train_from_iterator(texts, trainer)

    # Save
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    tokenizer.save(str(save_path))

    print(f"{model_type.upper()} tokenizer trained and saved to {save_path}")
    return tokenizer# https://arxiv.org/pdf/1909.03341 
# Expected output: ... Tokens will be segments of the base character set, often starting with 'G' or similar for spaces.

## Tokenizer training

In [14]:

bytelevel_tokenizer = train_custom_tokenizer(
    texts=text,
    model_type="bytelevel",
    vocab_size=16384,  
    save_path="tokenizers/bytelevel_tokenizer.json",
)

bpe_tokenizer = train_custom_tokenizer(
    texts=text,
    model_type="bpe",
    vocab_size=16384,  
    save_path="tokenizers/bpe_tokenizer.json",
)

unigram_tokenizer = train_custom_tokenizer(
    texts=text,
    model_type="unigram",
    vocab_size=16384,  
    save_path="tokenizers/unigram_tokenizer.json",
)

wordpiece_tokenizer = train_custom_tokenizer(
    texts=text,
    model_type="wordpiece",
    vocab_size=16384,  
    save_path="tokenizers/wordpiece_tokenizer.json",
)

tokenizers = {
    "bytelevel": bytelevel_tokenizer,
    "bpe": bpe_tokenizer,
    "unigram": unigram_tokenizer,
    "wordpiece": wordpiece_tokenizer,
}


Training BYTELEVEL tokenizer...



BYTELEVEL tokenizer trained and saved to tokenizers/bytelevel_tokenizer.json
Training BPE tokenizer...



BPE tokenizer trained and saved to tokenizers/bpe_tokenizer.json
Training UNIGRAM tokenizer...


UNIGRAM tokenizer trained and saved to tokenizers/unigram_tokenizer.json
Training WORDPIECE tokenizer...



WORDPIECE tokenizer trained and saved to tokenizers/wordpiece_tokenizer.json


In [15]:
# --------------------------------------------------
#                 Pre-trained tokenizer
# --------------------------------------------------

# choice of pre-trained tokenizer : "cl100k" "core Language" byte paired encoding (BPE), 100K vocab length
# BPE ensures no UNK, and full coverage of Unicode characters including Cyrillic 

import tiktoken
from types import SimpleNamespace

# -------------------------------
#     Pre-trained tokenizer
# -------------------------------

# Load the pre-trained tokenizer
enc = tiktoken.get_encoding("cl100k_base")

# adding a wrapper to make it compatible with custom tokenizers pipeline
class TiktokenWrapper:
    def __init__(self, enc):
        self.encoder = enc

    def encode(self, text: str):
        ids = self.encoder.encode(text)
        tokens = [self.encoder.decode([i]) for i in ids]
        return SimpleNamespace(ids=ids, tokens=tokens)
    
    def decode(self, ids: List[int]):
        return self.encoder.decode(ids)
    
    def token_to_id(self, token: str):
        ids = self.encoder.encode(token)
        if len(ids) == 1:
            return ids[0]
        return None
    
# creating the wrapped tokenizer
pretrained_tokenizer = TiktokenWrapper(enc)

# add to tokenizers dict
# -------------------------------
tokenizers["cl100k_pretrained"] = pretrained_tokenizer
# sanity check: cl100k_base tokenizer is added to Tokenizer dict
print("Registered tokenizers:", list(tokenizers.keys()))

# Test on your text
# -------------------------------
tokens = enc.encode("Привет, как дела?")
print(tokens)
print(enc.decode(tokens))

Registered tokenizers: ['bytelevel', 'bpe', 'unigram', 'wordpiece', 'cl100k_pretrained']
[54745, 28089, 8341, 11, 52770, 95369, 1506, 30]
Привет, как дела?


## Define metrics


### Tokenizer-only metrics
| Metric                               | Category     | Why tokenizer-only?                                       |
| ------------------------------------ | ------------ | --------------------------------------------------------- |
| **(3) Tokens / Character**           | Efficiency   | Pure tokenization behavior.                               |
| **(4) Average Characters / Token**   | Efficiency   | Depends only on segmentation.                             |
| **(5) Sequence Length Distribution** | Compute cost | Sequence length is produced by tokenizer before training. |
| **(7) OOV Rate**                     | Robustness   | Tokenizer’s ability to cover text.                        |
| **(8) Unicode / Character Coverage** | Robustness   | Tokenizer’s vocabulary coverage of character set.         |
| **(10) Vocab File Size**             | Storage      | Pure tokenizer metadata.                                  |


In [16]:

def tokens_per_character(tokenizer, texts):
    """
    Computes the ratio: total_tokens / total_characters
    Estimate tokenization efficiency --> A low value means the tokenizer is efficient
    """
    total_tokens = 0
    total_chars = 0
    for t in texts:
        enc = tokenizer.encode(t)
        # number of tokens
        total_tokens += len(enc.ids)
        # number of characters
        total_chars += len(t)

    value = total_tokens / max(total_chars, 1)
    print(f"[Tokens/Character] {value:.4f}")
    return value


def avg_characters_per_token(tokenizer, texts):
    """
    Measures how many characters each token represent on average
    Use the *lengths of token strings*, not raw text
    """
    total_chars = 0
    total_tokens = 0

    for t in texts:
        enc = tokenizer.encode(t)
        tokens = enc.tokens
        total_tokens += len(tokens)
        total_chars += sum(len(tok) for tok in tokens)

    value = total_chars / max(total_tokens, 1)
    print(f"[Avg Characters/Token] {value:.4f}")
    return value


# Computes basic statistics on tokenized sequence lengths
def sequence_length_distribution(tokenizer, texts, seq_len=128):
    """
    arg: tokenizer: the tokenizer to analyze
         texts: list of texts to analyze
         seq_len: the sequence length to consider
    returns: numpy array of sequence lengths
    """
    lengths = []

    for t in texts:
        enc = tokenizer.encode(t)
        lengths.append(len(enc.ids))

    lengths = np.array(lengths)

    print(f"[Sequence Length Distribution]")
    print(f"  mean: {lengths.mean():.2f}")
    print(f"  std:  {lengths.std():.2f}")
    print(f"  min:  {lengths.min():.2f}")
    print(f"  max:  {lengths.max():.2f}")
    return lengths

# number of unknown tokens: OOV = Out-of-Vocabulary
def oov_rate(tokenizer, texts):
    unk_id = tokenizer.token_to_id("[UNK]")
    if unk_id is None:
        print("[OOV Rate] this tokenizer has no [UNK] token")
        return 0.0

    total_tokens = 0
    unk_tokens = 0

    for t in texts:
        enc = tokenizer.encode(t)
        ids = enc.ids

        total_tokens += len(ids)
        # count UNK tokens
        unk_tokens += sum(1 for i in ids if i == unk_id)

    value = unk_tokens / max(total_tokens, 1)
    print(f"[OOV Rate] {value:.5f}")
    return value

# Unicode character coverage
def unicode_character_coverage(tokenizer, texts):
    """
    Percentage of UNIQUE characters in the dataset that can be
    encoded without producing UNK tokens
    --> relevant for our multilingual datasets
    """
    all_chars = set("".join(texts))
    covered_chars = 0

    # Check each character
    for ch in all_chars:
        enc = tokenizer.encode(ch)
        ids = enc.ids
        # If encoding the char results in UNK --> not covered
        if all(token_id != tokenizer.token_to_id("[UNK]") for token_id in ids):
            covered_chars += 1

    value = covered_chars / max(len(all_chars), 1)
    print(f"[Unicode Coverage] {value*100:.2f}%")
    return value


def analyze_tokenizer(tokenizer, texts, verbose=True):
    """
    Compute tokenizer statistics in a single pass for maximum efficiency
    Metrics:
        - tokens_per_character
        - avg_characters_per_token
        - sequence_length_distribution (min/mean/max/std)
        - OOV rate
        - unicode character coverage

    Arg: tokenizer: the tokenizer to analyze
         texts: list of texts to analyze
         verbose: whether to print the results
    returns: dictionary with all metrics
    """

    unk_id = tokenizer.token_to_id("[UNK]")

    # Cumulative statistics
    total_chars = 0           # sum of len(text)
    total_tokens = 0          # sum of all token counts
    total_unk = 0             # how many tokens became [UNK]
    seq_lengths = []          # list of sequence lengths

    all_chars = set("".join(texts))
    uncovered_chars = set()   # characters inside sequences that produced UNK

    for t in texts:
        # count raw characters
        total_chars += len(t)

        # tokenize text
        enc = tokenizer.encode(t)
        ids = enc.ids
        seq_len = len(ids)

        total_tokens += seq_len
        seq_lengths.append(seq_len)

        # Count UNK
        if unk_id is not None and unk_id in ids:
            total_unk += ids.count(unk_id)
            # mark characters as uncovered
            uncovered_chars.update(t)

    # ratio Tokens/Characters
    tokens_per_character = total_tokens / total_chars if total_chars > 0 else 0
    # average characters per token
    avg_characters_per_token = total_chars / total_tokens if total_tokens > 0 else 0
    seq_lengths = np.array(seq_lengths)

    # OOV rate --> fraction of tokens that are UNK
    oov_rate = total_unk / total_tokens if total_tokens > 0 else 0

    if unk_id is None:
        unicode_coverage = 1.0
    else:
        covered_chars = len(all_chars) - len(uncovered_chars)
        unicode_coverage = covered_chars / max(len(all_chars), 1)

    if verbose:
        print("\n=== Tokenizer Analysis ===")
        print(f"[Tokens/Character]          {tokens_per_character:.4f}")
        print(f"[Avg Characters/Token]      {avg_characters_per_token:.4f}")
        print(f"[Sequence Length Mean]      {seq_lengths.mean():.2f}")
        print(f"[Sequence Length Std]       {seq_lengths.std():.2f}")
        print(f"[Sequence Length Min]       {seq_lengths.min():.2f}")
        print(f"[Sequence Length Max]       {seq_lengths.max():.2f}")
        print(f"[OOV Rate]                  {oov_rate:.5f}")
        print(f"[Unicode Coverage]          {unicode_coverage*100:.2f}%")
        print("===========================\n")

    return {
        "tokens_per_character": tokens_per_character,
        "avg_characters_per_token": avg_characters_per_token,
        "sequence_lengths": seq_lengths,
        "sequence_length_mean": seq_lengths.mean(),
        "sequence_length_std": seq_lengths.std(),
        "sequence_length_min": seq_lengths.min(),
        "sequence_length_max": seq_lengths.max(),
        "oov_rate": oov_rate,
        "unicode_coverage": unicode_coverage,
    }

## Simple Encoding Example and Evaluation

In [17]:
# test run:
test_string = "This is a test string with English and русский текст."

print("--- Test Encodings with Sample String ---")
print(f"Test String: {test_string}")

# Encode and Decode test string
for name, tokenizer in tokenizers.items():
    enc = tokenizer.encode(test_string).ids
    dec = tokenizer.decode(enc)
    print(f"{name}: {dec}")
print()

# Evalate all our tokenizers
for name, tokenizer in tokenizers.items():
    print(f"--- Evaluating {name.upper()} Tokenizer ---")
    analyze_tokenizer(tokenizer, text)

--- Test Encodings with Sample String ---
Test String: This is a test string with English and русский текст.
bytelevel:  This is a test string with English and русский текст.
bpe: This is a test string with English and русский текст .
unigram: This is a test string with English and русски й текст .
wordpiece: This is a test string with English and русский текст .
cl100k_pretrained: This is a test string with English and русский текст.

--- Evaluating BYTELEVEL Tokenizer ---

=== Tokenizer Analysis ===
[Tokens/Character]          0.2776
[Avg Characters/Token]      3.6026
[Sequence Length Mean]      5260.78
[Sequence Length Std]       5953.80
[Sequence Length Min]       32.00
[Sequence Length Max]       50945.00
[OOV Rate]                  0.00000
[Unicode Coverage]          100.00%

--- Evaluating BPE Tokenizer ---

=== Tokenizer Analysis ===
[Tokens/Character]          0.2609
[Avg Characters/Token]      3.8335
[Sequence Length Mean]      4943.83
[Sequence Length Std]       5645.54
[Seq